# Message passing & Graph Convolutional Networks — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## A GCN layer averages neighbor information with a carefully normalized matrix
Message passing updates each node from nearby features. These examples move from raw sums to means, then to the symmetric GCN normalization with self-loops and a one-layer feature transform.

In [ ]:
# 1) raw neighbor sum messages
A=np.array([[0,1,1],[1,0,1],[1,1,0]],float); X=np.array([[1.,0.],[0.,1.],[1.,1.]])
M=A@X
plt.figure(figsize=(5,3)); plt.imshow(M,cmap='Blues'); plt.colorbar(); plt.title('raw neighbor sums')
assert np.allclose(M[0],[1,2])

In [ ]:
# 2) mean aggregation divides by degree
A=np.array([[0,1,1],[1,0,1],[1,1,0]],float); X=np.array([[1.,0.],[0.,1.],[1.,1.]])
Mean=(A@X)/A.sum(1,keepdims=True)
plt.figure(figsize=(5,3)); plt.imshow(Mean,cmap='Greens'); plt.colorbar(); plt.title('neighbor means')
assert np.allclose(Mean[0],[0.5,1])

In [ ]:
# 3) add self-loops and symmetric GCN normalization
A=np.array([[0,1,1],[1,0,1],[1,1,0]],float); I=np.eye(3); At=A+I; d=At.sum(1); S=np.diag(1/np.sqrt(d))@At@np.diag(1/np.sqrt(d))
plt.figure(figsize=(4,3)); plt.imshow(S,cmap='magma'); plt.colorbar(); plt.title('GCN S = D^-1/2 A~ D^-1/2')
assert np.allclose(S,np.ones((3,3))/3)

In [ ]:
# 4) one GCN layer: H = ReLU(S X W)
A=np.array([[0,1,1],[1,0,1],[1,1,0]],float); X=np.array([[1.,0.],[0.,1.],[1.,1.]]); W=np.array([[1.],[-1.]])
S=np.ones((3,3))/3; H=np.maximum(0,S@X@W)
plt.figure(figsize=(5,3)); plt.bar(range(3),H.ravel()); plt.title('one-layer GCN activations')
assert np.allclose(H.ravel(),[0,0,0])

In [ ]:
# 5) repeated averaging smooths node features
A=np.array([[0,1,0],[1,0,1],[0,1,0]],float); At=A+np.eye(3); d=At.sum(1); S=np.diag(1/np.sqrt(d))@At@np.diag(1/np.sqrt(d))
x=np.array([1.,0.,0.]); x2=S@S@x
plt.figure(figsize=(5,3)); plt.plot(x,'--o',label='start'); plt.plot(x2,'-o',label='after 2 layers'); plt.legend(); plt.title('two-hop smoothing')
assert x2[1]>0 and x2[2]>0